# 02 — Data Cleaning

## Mục tiêu
- Kiểm tra dữ liệu thô: missing values, kiểu dữ liệu, giá trị bất thường
- Chọn lọc các cột quan trọng, loại bỏ cột dư thừa (>90% missing)
- Chuyển đổi các cột text sang dạng numeric (diện tích, giá bán, số phòng ngủ)
- Chuẩn hóa giá trị categorical (giấy tờ pháp lý)
- Lưu dataset đã clean để sử dụng cho các bước tiếp theo


In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/dataset_merged.csv")


In [3]:
df.shape

(8273, 24)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8273 entries, 0 to 8272
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   tieu_de              7964 non-null   object 
 1   gia_ban              7961 non-null   object 
 2   don_gia              7961 non-null   object 
 3   dien_tich            7961 non-null   object 
 4   dia_chi              8273 non-null   object 
 5   mo_ta                7961 non-null   object 
 6   dien_thoai           3959 non-null   object 
 7   loai_hinh            7961 non-null   object 
 8   dien_tich_dat        7961 non-null   object 
 9   dien_tich_su_dung    3754 non-null   object 
 10  gia_m2               7961 non-null   object 
 11  giay_to_phap_ly      7460 non-null   object 
 12  so_phong_ngu         7961 non-null   object 
 13  so_phong_ve_sinh     5977 non-null   object 
 14  tong_so_tang         5501 non-null   float64
 15  tinh_trang_noi_that  4432 non-null   o

### Nhận xét

Dataset ban đầu có **8,273 dòng** và **24 cột**. Quan sát sơ bộ:
- Tất cả các cột đều ở dạng **object** (text) trừ  (float64)
- Nhiều cột bị thiếu dữ liệu nghiêm trọng:  (25/8273),  (25/8273),  (1321/8273)
- Các cột quan trọng (, , ) có ~7,961 giá trị → ~312 dòng null (~3.8%)


In [5]:
df.columns

Index(['tieu_de', 'gia_ban', 'don_gia', 'dien_tich', 'dia_chi', 'mo_ta',
       'dien_thoai', 'loai_hinh', 'dien_tich_dat', 'dien_tich_su_dung',
       'gia_m2', 'giay_to_phap_ly', 'so_phong_ngu', 'so_phong_ve_sinh',
       'tong_so_tang', 'tinh_trang_noi_that', 'huong_cua_chinh', 'dac_diem',
       'chieu_ngang', 'chieu_dai', 'ma_can', 'ten_phan_khu_lo', 'bieu_do_gia',
       'source'],
      dtype='object')

In [6]:
missing = df.isnull().sum().sort_values(ascending=False)
missing

ten_phan_khu_lo        8248
ma_can                 8248
huong_cua_chinh        6952
dien_tich_su_dung      4519
dien_thoai             4314
tinh_trang_noi_that    3841
dac_diem               3066
tong_so_tang           2772
chieu_dai              2303
so_phong_ve_sinh       2296
chieu_ngang            2167
giay_to_phap_ly         813
don_gia                 312
dien_tich               312
gia_ban                 312
loai_hinh               312
so_phong_ngu            312
gia_m2                  312
mo_ta                   312
dien_tich_dat           312
tieu_de                 309
dia_chi                   0
bieu_do_gia               0
source                    0
dtype: int64

In [7]:
missing_percent = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_percent


ten_phan_khu_lo        99.697812
ma_can                 99.697812
huong_cua_chinh        84.032395
dien_tich_su_dung      54.623474
dien_thoai             52.145534
tinh_trang_noi_that    46.428140
dac_diem               37.060317
tong_so_tang           33.506588
chieu_dai              27.837544
so_phong_ve_sinh       27.752931
chieu_ngang            26.193642
giay_to_phap_ly         9.827149
don_gia                 3.771304
dien_tich               3.771304
gia_ban                 3.771304
loai_hinh               3.771304
so_phong_ngu            3.771304
gia_m2                  3.771304
mo_ta                   3.771304
dien_tich_dat           3.771304
tieu_de                 3.735042
dia_chi                 0.000000
bieu_do_gia             0.000000
source                  0.000000
dtype: float64

### Nhận xét Missing Values
- **>99% missing**: `ma_can`, `ten_phan_khu_lo` → gần như không có dữ liệu, loại bỏ
- **>80% missing**: `huong_cua_chinh` (84%) → loại bỏ
- **>50% missing**: `dien_tich_su_dung` (54.6%), `dien_thoai` (52.1%) → loại bỏ
- **33-47% missing**: `tinh_trang_noi_that`, `dac_diem`, `tong_so_tang` → loại bỏ
- **~3.8% missing**: Các cột chính (`gia_ban`, `dien_tich`, `loai_hinh`,...) → xóa dòng null
- **0% missing**: `dia_chi`, `bieu_do_gia`, `source` → giữ nguyên


In [8]:
df[(~df['tong_so_tang'].isnull())][[
    "chieu_ngang",
    "chieu_dai",
    "dien_tich",
    "dien_tich_dat",
    "dien_tich_su_dung",
    "gia_m2",
    "don_gia",
    "gia_ban",
    "so_phong_ngu",
    "tong_so_tang",
    "so_phong_ve_sinh",
    "dac_diem",
    "tinh_trang_noi_that",
    "giay_to_phap_ly",
    "loai_hinh",
    "dia_chi",
    # "ten_phan_khu_lo",
    # "ma_can",
    # "huong_cua_chinh",
    # "dien_thoai",
    # "mo_ta",
    # "tieu_de",
    # "bieu_do_gia",
    # "source"
]].head(5)

,chieu_ngang,chieu_dai,dien_tich,dien_tich_dat,dien_tich_su_dung,gia_m2,don_gia,gia_ban,so_phong_ngu,tong_so_tang,so_phong_ve_sinh,dac_diem,tinh_trang_noi_that,giay_to_phap_ly,loai_hinh,dia_chi
0,4.5 m,8 m,36 m²,36 m²,NaN,"106,94 triệu/m²","106,94 triệu/m²","3,85 tỷ",2 phòng,2.0,2 phòng,Nhà nở hậu,Nội thất đầy đủ,Đã có sổ,"Nhà ngõ, hẻm","Đường Lê Quang Định, Phường 7, Quận Bình Thạnh..."
2,7 m,7.7 m,54 m²,54 m²,NaN,"133,33 triệu/m²","133,33 triệu/m²","7,2 tỷ",3 phòng,2.0,2 phòng,Hẻm xe hơi,Hoàn thiện cơ bản,Đã có sổ,"Nhà ngõ, hẻm","Đường Nơ Trang Long, Phường 13, Quận Bình Thạn..."
5,5.6 m,21 m,120 m²,120 m²,166 m²,"108,33 triệu/m²","108,33 triệu/m²",13 tỷ,5 phòng,2.0,3 phòng,NaN,NaN,Đã có sổ,"Nhà ngõ, hẻm","Đường Bạch Đằng, Phường 15, Quận Bình Thạnh, T..."
6,4.2 m,11.3 m,59.4 m²,59.4 m²,59.4 m²,"188,55 triệu/m²","188,55 triệu/m²","11,2 tỷ",4 phòng,3.0,4 phòng,Hẻm xe hơi,NaN,Đã có sổ,"Nhà ngõ, hẻm","Đường Nguyễn Xí, Phường 26, Quận Bình Thạnh, T..."
8,4 m,20 m,80 m²,80 m²,320 m²,"322,50 triệu/m²","322,50 triệu/m²","25,8 tỷ",9 phòng,5.0,5 phòng,NaN,Nội thất đầy đủ,Đã có sổ,"Nhà mặt phố, mặt tiền","89, Đường Nguyễn Văn Thương, Phường 25, Quận B..."


In [9]:
df[(df['gia_m2'] != df['don_gia']) & df['don_gia'].notnull()][[
    "chieu_ngang",
    "chieu_dai",
    "dien_tich",
    "dien_tich_dat",
    "dien_tich_su_dung",
    "gia_m2",
    "don_gia",
    "gia_ban",
    "so_phong_ngu",
    "tong_so_tang",
    "so_phong_ve_sinh",
    "dac_diem",
    "tinh_trang_noi_that",
    "giay_to_phap_ly",
    "loai_hinh",
    "dia_chi",
    # "ten_phan_khu_lo",
    # "ma_can",
    # "huong_cua_chinh",
    # "dien_thoai",
    # "mo_ta",
    # "tieu_de",
    # "bieu_do_gia",
    # "source"
]].head(5)

,chieu_ngang,chieu_dai,dien_tich,dien_tich_dat,dien_tich_su_dung,gia_m2,don_gia,gia_ban,so_phong_ngu,tong_so_tang,so_phong_ve_sinh,dac_diem,tinh_trang_noi_that,giay_to_phap_ly,loai_hinh,dia_chi


In [10]:
df[(df['dien_tich'] != df['dien_tich_dat']) & df['dien_tich_dat'].notnull()][[
    "chieu_ngang",
    "chieu_dai",
    "dien_tich",
    "dien_tich_dat",
    "dien_tich_su_dung",
    "gia_m2",
    "don_gia",
    "gia_ban",
    "so_phong_ngu",
    "tong_so_tang",
    "so_phong_ve_sinh",
    "dac_diem",
    "tinh_trang_noi_that",
    "giay_to_phap_ly",
    "loai_hinh",
    "dia_chi",
    # "ten_phan_khu_lo",
    # "ma_can",
    # "huong_cua_chinh",
    # "dien_thoai",
    # "mo_ta",
    # "tieu_de",
    # "bieu_do_gia",
    # "source"
]].head(5)

,chieu_ngang,chieu_dai,dien_tich,dien_tich_dat,dien_tich_su_dung,gia_m2,don_gia,gia_ban,so_phong_ngu,tong_so_tang,so_phong_ve_sinh,dac_diem,tinh_trang_noi_that,giay_to_phap_ly,loai_hinh,dia_chi


In [11]:
df[
    (df['dien_tich'].isnull())
    & (df['gia_ban'].isnull())
    & (df['loai_hinh'].isnull())
    & (df['so_phong_ngu'].isnull())
    & (df['gia_m2'].isnull())
    & (df['mo_ta'].isnull())
    # & (df['tieu_de'].isnull())
].count()

tieu_de                  3
gia_ban                  0
don_gia                  0
dien_tich                0
dia_chi                312
mo_ta                    0
dien_thoai               0
loai_hinh                0
dien_tich_dat            0
dien_tich_su_dung        0
gia_m2                   0
giay_to_phap_ly          0
so_phong_ngu             0
so_phong_ve_sinh         0
tong_so_tang             0
tinh_trang_noi_that      0
huong_cua_chinh          0
dac_diem                 0
chieu_ngang              0
chieu_dai                0
ma_can                   0
ten_phan_khu_lo          0
bieu_do_gia            312
source                 312
dtype: int64

## 1. Chọn lọc cột quan trọng


```plaintext
keep_cols = [
    # "ten_phan_khu_lo        99.697812", --> Thiếu nhiều quá
    # "ma_can                 99.697812", --> Thiếu nhiều quá
    # "huong_cua_chinh        84.032395", --> Thiếu nhiều quá
    # "dien_tich_su_dung      54.623474", --> Thiếu nhiều quá, không thể tính median hay mean được do không có căn cứ
    # "dien_thoai             52.145534", --> Không liên quan
    # "don_gia                 3.771304", --> Có gia_m2 rồi
    # "dien_tich_dat           3.771304", --> Có dien_tich rồi
    # "source                  0.000000", --> không liên quan
    "tinh_trang_noi_that    46.428140", # có thể fill là unknown
    # "chieu_dai              27.837544", --> Là thông số tính dien_tich, không thể fill na được
    # "dac_diem               37.060317", --> Có thể trích từ mo_ta
    # "chieu_ngang            26.193642", --> Là thông số tính dien_tich, không thể fill na được
    "tong_so_tang           33.506588", # na có thể fill là 1 hoặc 2, nhưng sẽ không chính xác, nên bỏ luôn cột này
    "so_phong_ve_sinh       27.752931", # na có thể fill là 1 hoặc 2, nhưng sẽ không chính xác, nên bỏ luôn cột này
    "giay_to_phap_ly         9.827149", # có thể fill là unknown
    "dien_tich               3.771304",
    "gia_ban                 3.771304",
    "loai_hinh               3.771304",
    "so_phong_ngu            3.771304",
    "gia_m2                  3.771304", # gia_2 có thể suy từ gia_ban / dien_tich
    "mo_ta                   3.771304",
    "tieu_de                 3.735042",
    "dia_chi                 0.000000",
    "bieu_do_gia             0.000000",
]
```

In [12]:
keep_cols = [
    "giay_to_phap_ly",
    "dien_tich",
    "gia_ban",
    "loai_hinh",
    "so_phong_ngu",
    "mo_ta",
    "tieu_de",
    "dia_chi",
    "bieu_do_gia",
]

# 

In [13]:
df = df[keep_cols]

In [14]:
df.shape

(8273, 9)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8273 entries, 0 to 8272
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   giay_to_phap_ly  7460 non-null   object
 1   dien_tich        7961 non-null   object
 2   gia_ban          7961 non-null   object
 3   loai_hinh        7961 non-null   object
 4   so_phong_ngu     7961 non-null   object
 5   mo_ta            7961 non-null   object
 6   tieu_de          7964 non-null   object
 7   dia_chi          8273 non-null   object
 8   bieu_do_gia      8273 non-null   object
dtypes: object(9)
memory usage: 581.8+ KB


In [16]:
df.head()

,giay_to_phap_ly,dien_tich,gia_ban,loai_hinh,so_phong_ngu,mo_ta,tieu_de,dia_chi,bieu_do_gia
0,Đã có sổ,36 m²,"3,85 tỷ","Nhà ngõ, hẻm",2 phòng,🍀VÂN KIỀU AN CƯ🍀\r\n\r\n🏡 Nhà Lê Quang Định P5...,🍀VÂN KIỀU AN CƯ🍀 40m2 - Hẻm Ô Tô - Gần mặt tiề...,"Đường Lê Quang Định, Phường 7, Quận Bình Thạnh...","[134.04, 125.8, 125.8, 125.8, 138.64, 138.64, ..."
1,NaN,62 m²,"9,79 tỷ","Nhà ngõ, hẻm",4 phòng,"CHỈ 9.XTY - NHÀ MỚI KENG,XE HƠI NGỦ NHÀ GIÁP B...","9.xty,Nhà Hẻm Xe Hơi,Xây Mới Khu VIP giáp Phạm...","Đường Phạm Văn Đồng, Phường 13, Quận Bình Thạn...","[133.33, 135.54, 130.95, 130.02, 130.02, 136.3..."
2,Đã có sổ,54 m²,"7,2 tỷ","Nhà ngõ, hẻm",3 phòng,XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,"Đường Nơ Trang Long, Phường 13, Quận Bình Thạn...","[168.37, 168.37, 120.71, 120.71, 124.24, 124.5..."
3,Đã có sổ,83 m²,"8,5 tỷ","Nhà ngõ, hẻm",3 phòng,#e45\r\n🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TI...,🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TIỆN XÂY M...,"Đường Xô Viết Nghệ Tĩnh, Phường 21, Quận Bình ...","[159.22, 159.22, 157.45, 157.45, 135.08, 122.2..."
4,Đã có sổ,18 m²,"2,85 tỷ","Nhà ngõ, hẻm",2 phòng,Nhà hẻm cách đường mặt tiền 20m\r\nNhà ở khu v...,Nhà hẻm cách đường mặt tiền 20m,"Đường Chu Văn An, Phường 12, Quận Bình Thạnh, ...","[132.67, 132.67, 169.64, 169.64, 133.33, 133.3..."


## 2. Chuẩn hóa cột 


In [17]:
df["giay_to_phap_ly"].unique()

array(['Đã có sổ', nan, 'Đang chờ sổ', 'Sổ chung / công chứng vi bằng',
       'Giấy tờ viết tay', 'Không có sổ'], dtype=object)

In [18]:
df["giay_to_phap_ly"] = df["giay_to_phap_ly"].str.lower()

In [19]:
df["giay_to_phap_ly"].head()

0    đã có sổ
1         NaN
2    đã có sổ
3    đã có sổ
4    đã có sổ
Name: giay_to_phap_ly, dtype: object

In [20]:
df.loc[df["giay_to_phap_ly"].isnull(), "giay_to_phap_ly"] = "chưa xác định"

In [21]:
df["giay_to_phap_ly"].isnull().sum()

np.int64(0)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8273 entries, 0 to 8272
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   giay_to_phap_ly  8273 non-null   object
 1   dien_tich        7961 non-null   object
 2   gia_ban          7961 non-null   object
 3   loai_hinh        7961 non-null   object
 4   so_phong_ngu     7961 non-null   object
 5   mo_ta            7961 non-null   object
 6   tieu_de          7964 non-null   object
 7   dia_chi          8273 non-null   object
 8   bieu_do_gia      8273 non-null   object
dtypes: object(9)
memory usage: 581.8+ KB


In [23]:
df = df[
    ~(
        (df['dien_tich'].isnull())
        & (df['gia_ban'].isnull())
        & (df['loai_hinh'].isnull())
        & (df['so_phong_ngu'].isnull())
        & (df['mo_ta'].isnull())
    )
]

In [24]:
df.shape

(7961, 9)

### Nhận xét
Sau khi xóa 312 dòng thiếu thông tin chính, dataset còn **7,961 dòng** với **9 cột** đã chọn.


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7961 entries, 0 to 8272
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   giay_to_phap_ly  7961 non-null   object
 1   dien_tich        7961 non-null   object
 2   gia_ban          7961 non-null   object
 3   loai_hinh        7961 non-null   object
 4   so_phong_ngu     7961 non-null   object
 5   mo_ta            7961 non-null   object
 6   tieu_de          7961 non-null   object
 7   dia_chi          7961 non-null   object
 8   bieu_do_gia      7961 non-null   object
dtypes: object(9)
memory usage: 622.0+ KB


## 3. Chuyển đổi cột  sang numeric


In [26]:
df['dien_tich'].unique()

array(['36 m²', '62 m²', '54 m²', '83 m²', '18 m²', '120 m²', '59.4 m²',
       '26 m²', '80 m²', '68 m²', '60 m²', '200 m²', '88 m²', '48 m²',
       '70 m²', '130 m²', '303 m²', '48.9 m²', '92 m²', '20 m²', '40 m²',
       '72 m²', '34 m²', '110 m²', '32 m²', '30 m²', '63.6 m²', '17 m²',
       '35 m²', '27 m²', '50 m²', '56 m²', '78 m²', '44 m²', '14 m²',
       '24 m²', '46 m²', '38.7 m²', '52 m²', '105 m²', '90 m²', '35.5 m²',
       '53 m²', '42.2 m²', '22 m²', '19 m²', '55 m²', '64 m²', '33 m²',
       '41.3 m²', '45 m²', '31 m²', '43 m²', '87 m²', '15 m²', '67 m²',
       '125 m²', '47 m²', '25 m²', '41 m²', '81 m²', '38.8 m²', '79 m²',
       '12.5 m²', '22.8 m²', '115 m²', '16.2 m²', '73.6 m²', '82 m²',
       '69 m²', '300 m²', '74 m²', '75 m²', '57 m²', '28 m²', '63 m²',
       '155 m²', '58 m²', '32.4 m²', '85 m²', '37 m²', '52.6 m²',
       '34.5 m²', '88.5 m²', '161 m²', '68.1 m²', '76 m²', '27.5 m²',
       '38.1 m²', '73 m²', '96 m²', '45.7 m²', '39 m²', '66 m²',
     

In [27]:
def transform_dien_tich(value):
    if pd.isnull(value):
        return value
    if isinstance(value, str):
        value = value.replace("m2", "").replace("m²", "").strip()
        try:
            return float(value)
        except ValueError:
            return np.nan
    return value

df['dien_tich'] = df['dien_tich'].apply(transform_dien_tich)

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7961 entries, 0 to 8272
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   giay_to_phap_ly  7961 non-null   object 
 1   dien_tich        7961 non-null   float64
 2   gia_ban          7961 non-null   object 
 3   loai_hinh        7961 non-null   object 
 4   so_phong_ngu     7961 non-null   object 
 5   mo_ta            7961 non-null   object 
 6   tieu_de          7961 non-null   object 
 7   dia_chi          7961 non-null   object 
 8   bieu_do_gia      7961 non-null   object 
dtypes: float64(1), object(8)
memory usage: 622.0+ KB


### Nhận xét
Cột  đã được chuyển từ dạng text (vd: "36 m²") sang **float64** thành công. Các bước xử lý:
- Loại bỏ ký tự "m2", "m²"
- Parse sang float
- Giá trị không parse được → NaN


In [29]:
df.head()

,giay_to_phap_ly,dien_tich,gia_ban,loai_hinh,so_phong_ngu,mo_ta,tieu_de,dia_chi,bieu_do_gia
0,đã có sổ,36.0,"3,85 tỷ","Nhà ngõ, hẻm",2 phòng,🍀VÂN KIỀU AN CƯ🍀\r\n\r\n🏡 Nhà Lê Quang Định P5...,🍀VÂN KIỀU AN CƯ🍀 40m2 - Hẻm Ô Tô - Gần mặt tiề...,"Đường Lê Quang Định, Phường 7, Quận Bình Thạnh...","[134.04, 125.8, 125.8, 125.8, 138.64, 138.64, ..."
1,chưa xác định,62.0,"9,79 tỷ","Nhà ngõ, hẻm",4 phòng,"CHỈ 9.XTY - NHÀ MỚI KENG,XE HƠI NGỦ NHÀ GIÁP B...","9.xty,Nhà Hẻm Xe Hơi,Xây Mới Khu VIP giáp Phạm...","Đường Phạm Văn Đồng, Phường 13, Quận Bình Thạn...","[133.33, 135.54, 130.95, 130.02, 130.02, 136.3..."
2,đã có sổ,54.0,"7,2 tỷ","Nhà ngõ, hẻm",3 phòng,XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,"Đường Nơ Trang Long, Phường 13, Quận Bình Thạn...","[168.37, 168.37, 120.71, 120.71, 124.24, 124.5..."
3,đã có sổ,83.0,"8,5 tỷ","Nhà ngõ, hẻm",3 phòng,#e45\r\n🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TI...,🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TIỆN XÂY M...,"Đường Xô Viết Nghệ Tĩnh, Phường 21, Quận Bình ...","[159.22, 159.22, 157.45, 157.45, 135.08, 122.2..."
4,đã có sổ,18.0,"2,85 tỷ","Nhà ngõ, hẻm",2 phòng,Nhà hẻm cách đường mặt tiền 20m\r\nNhà ở khu v...,Nhà hẻm cách đường mặt tiền 20m,"Đường Chu Văn An, Phường 12, Quận Bình Thạnh, ...","[132.67, 132.67, 169.64, 169.64, 133.33, 133.3..."


## 4. Chuyển đổi cột  sang numeric


In [30]:
df["gia_ban"].unique()

array(['3,85 tỷ', '9,79 tỷ', '7,2 tỷ', '8,5 tỷ', '2,85 tỷ', '13 tỷ',
       '11,2 tỷ', '4,58 tỷ', '25,8 tỷ', '11,3 tỷ', '11,5 tỷ', '24,5 tỷ',
       '33,9 tỷ', '4,7 tỷ', '9,3 tỷ', '10,8 tỷ', '62 tỷ', '8 tỷ', '11 tỷ',
       '6,4 tỷ', '5,959 tỷ', '25,2 tỷ', '3,68 tỷ', '5,98 tỷ', '7 tỷ',
       '3,98 tỷ', '4,68 tỷ', '22 tỷ', '3,9 tỷ', '6,25 tỷ', '7,3 tỷ',
       '6,19 tỷ', '7,89 tỷ', '7,79 tỷ', '5,5 tỷ', '2,75 tỷ', '5,65 tỷ',
       '6,5 tỷ', '5 tỷ', '5,45 tỷ', '11,9 tỷ', '6,868 tỷ', '7,5 tỷ',
       '9,5 tỷ', '9,6 tỷ', '7,6 tỷ', '10,29 tỷ', '4 tỷ', '15 tỷ',
       '3,55 tỷ', '7,9 tỷ', '6,95 tỷ', '8,45 tỷ', '18 tỷ', '14,5 tỷ',
       '6,8 tỷ', '8,2 tỷ', '2,7 tỷ', '5,8 tỷ', '10,1 tỷ', '3,8 tỷ',
       '10 tỷ', '11,8 tỷ', '2,9 tỷ', '2,35 tỷ', '7,88 tỷ', '7,59 tỷ',
       '19,5 tỷ', '7,65 tỷ', '7,4 tỷ', '16 tỷ', '15,999 tỷ', '4,59 tỷ',
       '22,5 tỷ', '5,35 tỷ', '6 tỷ', '11,98 tỷ', '7,8 tỷ', '2,34 tỷ',
       '3,94 tỷ', '12 tỷ', '6,48 tỷ', '5,2 tỷ', '9,99 tỷ', '1,9 tỷ',
       '4,95 tỷ', 

In [31]:
def transform_gia_ban(value):
    if pd.isnull(value):
        return value
    if isinstance(value, str):
        if "tỷ" in value:
            value = value.replace("tỷ", "").replace(",", ".").strip()
            try:
                return float(value)
            except ValueError:
                return np.nan
        elif "triệu" in value:
            value = value.replace("triệu", "").replace(",", ".").strip()
            try:
                return float(value) / 1000  # Chuyển triệu sang tỷ
            except ValueError:
                return np.nan
    return value

df['gia_ban'] = df['gia_ban'].apply(transform_gia_ban)

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7961 entries, 0 to 8272
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   giay_to_phap_ly  7961 non-null   object 
 1   dien_tich        7961 non-null   float64
 2   gia_ban          7961 non-null   float64
 3   loai_hinh        7961 non-null   object 
 4   so_phong_ngu     7961 non-null   object 
 5   mo_ta            7961 non-null   object 
 6   tieu_de          7961 non-null   object 
 7   dia_chi          7961 non-null   object 
 8   bieu_do_gia      7961 non-null   object 
dtypes: float64(2), object(7)
memory usage: 622.0+ KB


### Nhận xét
Cột  đã được chuyển từ dạng text sang **float64** (đơn vị: tỷ VNĐ). Logic chuyển đổi:
- "X tỷ" → X (float)
- "X triệu" → X / 1000 (quy về tỷ)
- Giá trị không parse được → NaN


## 5. Chuyển đổi cột  sang numeric


In [33]:
df["so_phong_ngu"].unique()

array(['2 phòng', '4 phòng', '3 phòng', '5 phòng', '9 phòng', '6 phòng',
       'nhiều hơn 10 phòng', '7 phòng', '1 phòng', '10 phòng', '8 phòng'],
      dtype=object)

In [34]:
def transform_so_phong_ngu(value):
    if pd.isnull(value):
        return value
    if isinstance(value, str):
        if value.lower() in ["nhiều hơn 10 phòng"]:
            return 11  # Giả sử "nhiều hơn 10 phòng" là 11
        value = value.replace("phòng", "").strip()
        try:
            return int(value)
        except ValueError:
            return np.nan
    return value

df['so_phong_ngu'] = df['so_phong_ngu'].apply(transform_so_phong_ngu)

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7961 entries, 0 to 8272
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   giay_to_phap_ly  7961 non-null   object 
 1   dien_tich        7961 non-null   float64
 2   gia_ban          7961 non-null   float64
 3   loai_hinh        7961 non-null   object 
 4   so_phong_ngu     7961 non-null   int64  
 5   mo_ta            7961 non-null   object 
 6   tieu_de          7961 non-null   object 
 7   dia_chi          7961 non-null   object 
 8   bieu_do_gia      7961 non-null   object 
dtypes: float64(2), int64(1), object(6)
memory usage: 622.0+ KB


### Nhận xét
Cột  đã được chuyển sang **int64**. Xử lý đặc biệt:
- "X phòng" → X (int)
- "nhiều hơn 10 phòng" → 11 (giả định hợp lý, chỉ chiếm rất ít bản ghi)
- Giá trị không parse được → NaN


In [36]:
df.shape

(7961, 9)

## 6. Lưu dataset đã clean


In [37]:
df.to_csv("../data/processed/dataset_cleaned.csv", index=False, encoding="utf-8-sig")

In [38]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [39]:
df.to_parquet("../data/processed/dataset_cleaned.parquet", engine="pyarrow", index=False)


In [40]:
import os
os.listdir("../data/processed/")

['dataset_cleaned.csv',
 'dataset_cleaned.parquet',
 'dataset_merged.csv',
 'data_modeling.csv',
 'data_modeling.parquet',
 'pyspark_predictions.parquet']

In [41]:
test_df = pd.read_csv("../data/processed/dataset_cleaned.csv", encoding="utf-8-sig")

In [42]:
test_df.shape

(7961, 9)

In [43]:
test_df = pd.read_parquet("../data/processed/dataset_cleaned.parquet", engine="pyarrow")

In [44]:
test_df.shape

(7961, 9)

## Kết luận

### Tổng kết quá trình Data Cleaning

| Bước | Mô tả | Trước | Sau |
|------|-------|-------|-----|
| Load data | Dataset thô từ notebook 01 | 8,273 x 24 | - |
| Chọn cột | Giữ 9 cột quan trọng, bỏ 15 cột dư thừa | 24 cột | 9 cột |
| Xóa dòng rỗng | Loại bỏ dòng thiếu toàn bộ thông tin | 8,273 dòng | 7,961 dòng |
| Clean  | Lowercase + fill NaN = "chưa xác định" | 501 NaN | 0 NaN |
| Clean  | Text → float (m²) | object | float64 |
| Clean  | Text → float (tỷ VNĐ) | object | float64 |
| Clean  | Text → int | object | int64 |

### Dataset sau cleaning
- **Số dòng**: 7,961 (giảm 312 dòng, ~3.8%)
- **Số cột**: 9
- **Không còn missing values** ở các cột quan trọng
- **Kiểu dữ liệu** đã chuẩn hóa: 2 float, 1 int, 6 object

### Các cột đã bỏ và lý do
- , : >99% missing
- : >84% missing
- : >52% missing, không liên quan đến giá
- , , : >33% missing
- , : ~26-28% missing, thông tin đã có trong 
- , , , : trùng lặp hoặc dẫn xuất từ các cột khác
- : chỉ dùng cho truy vết, không cần cho modeling

### File output
-  (CSV, UTF-8-sig)
-  (Parquet, PyArrow)

### Bước tiếp theo
→ Notebook : Trích xuất thêm features từ text (tiêu đề, mô tả, địa chỉ) và biểu đồ giá khu vực.
